# 02a Retrain EMA Checkpoints

Run this notebook on Colab A100 High-RAM. Colab packages are scoped to one runtime session, so this notebook replays the canonical base-dependency and Mamba bootstrap from Notebook 02 when necessary. A dependency or Torch pin may intentionally restart Colab; after reconnecting, rerun Notebook 02a from the first cell. It retrains the five Chapman folds and writes explicit EMA/raw checkpoint variants to a versioned Drive model run directory, not the historical `Drive/ECG-Ramba/model` directory. `fold*_final_ema.pt` at the pre-specified epoch is the manuscript OOF checkpoint; `fold*_best_ema.pt` is retained only as a validation-selected diagnostic. Do not continue to Notebook 02 until both variants pass verification and the current model-run pointer is written.


## Setup Repository And Drive Paths


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
MODEL_RUNS_DIR = DRIVE_ROOT / 'model_runs'
RETRAIN_EPOCHS = int(os.environ.get('ECG_RAMBA_RETRAIN_EPOCHS', '20'))
DEFAULT_MODEL_RUN_ID = f'ema_protocol_e{RETRAIN_EPOCHS}_v2'
MODEL_RUN_ID = os.environ.get('ECG_RAMBA_RETRAIN_RUN_ID', DEFAULT_MODEL_RUN_ID)
MODEL_DIR = Path(os.environ.get('ECG_RAMBA_MODEL_DIR', str(MODEL_RUNS_DIR / MODEL_RUN_ID))).expanduser()
LEGACY_MODEL_DIR = DRIVE_ROOT / 'model'
CURRENT_MODEL_POINTER = MODEL_RUNS_DIR / 'current_final_ema_model_dir.txt'
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path(os.environ.get('ECG_RAMBA_LOCAL_ROOT', '/content/ecg_ramba_runtime'))
LOCAL_EXTRACT_DIR = Path(os.environ.get('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman')))

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

from google.colab import drive
if not _drive_root_ready(DRIVE_ROOT):
    try:
        drive.mount(str(DRIVE_MOUNT))
    except Exception as exc:
        print(f'Drive mount initial attempt failed or was stale: {exc}')
        drive.mount(str(DRIVE_MOUNT), force_remount=True)
else:
    print('Drive root already visible:', DRIVE_ROOT)
if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')

def run(cmd, *, cwd=None, check=True, log_path=None):
    print('$', cmd, flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        return subprocess.run(cmd, shell=True, cwd=str(run_cwd), check=check)
    log_path = Path(log_path)
    if not log_path.is_absolute():
        log_path = run_cwd / log_path
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd, shell=True, cwd=str(run_cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return_code = proc.wait()
    print('Command log:', log_path)
    if check and return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
if MODEL_DIR.resolve() == LEGACY_MODEL_DIR.resolve():
    raise RuntimeError('Refusing to retrain into the historical Drive/ECG-Ramba/model directory. Set ECG_RAMBA_MODEL_DIR to a model_runs subdirectory.')

if not REPO_DIR.exists():
    run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
run('git fetch origin', cwd=REPO_DIR)
run(f'git checkout {BRANCH}', cwd=REPO_DIR)
run(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR)
os.chdir(REPO_DIR)

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['ECG_RAMBA_MODEL_DIR'] = str(MODEL_DIR)
os.environ['ECG_RAMBA_EPOCHS'] = str(RETRAIN_EPOCHS)
os.environ['ECG_RAMBA_LOCAL_ROOT'] = str(LOCAL_RUNTIME_ROOT)
os.environ['ECG_RAMBA_EXTRACT_DIR'] = str(LOCAL_EXTRACT_DIR)
os.environ['ECG_RAMBA_SAVE_CLEAN_CACHE'] = '1'
os.environ['ECG_RAMBA_USE_CLEAN_CACHE'] = '1'

print('cwd       :', Path.cwd())
print('drive root:', DRIVE_ROOT)
print('model dir :', MODEL_DIR)
print('local root:', LOCAL_RUNTIME_ROOT)
print('extract dir:', LOCAL_EXTRACT_DIR)
print('epochs    :', RETRAIN_EPOCHS)
print('legacy dir:', LEGACY_MODEL_DIR)
print('pointer   :', CURRENT_MODEL_POINTER)
print('branch    :', subprocess.check_output('git branch --show-current', shell=True, text=True).strip())
print('commit    :', subprocess.check_output('git rev-parse HEAD', shell=True, text=True).strip())
print('git status:', subprocess.check_output('git status --short --branch', shell=True, text=True).strip())


## Runtime Bootstrap And Sanity Check


In [ ]:
import importlib
import json

installer_nb_path = REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb'
installer_nb = json.loads(installer_nb_path.read_text(encoding='utf-8'))

def canonical_installer_source(*markers):
    for notebook_cell in installer_nb['cells']:
        if notebook_cell.get('cell_type') != 'code':
            continue
        source = ''.join(notebook_cell.get('source', []))
        if all(marker in source for marker in markers):
            return source
    raise RuntimeError(f'Could not locate canonical installer cell containing: {markers}')

print('Colab packages are runtime-scoped; validating the complete ECG-RAMBA environment for this session.')
base_installer = canonical_installer_source('INSTALL_BASE_DEPS = True', 'REPAIR_BROKEN_NUMERIC_STACK')
exec(compile(base_installer, str(installer_nb_path) + ':base-deps', 'exec'), globals(), globals())

import torch

print('Python:', sys.version)
print('Torch :', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Retraining requires a CUDA GPU. Select A100 High-RAM before running this notebook.')

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print('GPU RAM:', f'{gpu_mem_gb:.1f} GiB')
if gpu_mem_gb < 35:
    raise RuntimeError('Use A100 High-RAM for full five-fold retraining. T4/L4 can be too slow or memory-limited.')

required_modules = ['mamba_ssm', 'causal_conv1d']
module_errors = {}
for module in required_modules:
    try:
        importlib.import_module(module)
    except Exception as exc:
        module_errors[module] = repr(exc)

if module_errors:
    print('Mamba runtime requires installation/repair:', module_errors)
    model_installer = canonical_installer_source('INSTALL_MODEL_DEPS = True', 'AUTO_PIN_TORCH_FOR_MAMBA')
    exec(compile(model_installer, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
    importlib.invalidate_caches()

module_errors = {}
for module in required_modules:
    try:
        loaded = importlib.import_module(module)
        print('OK import:', module, getattr(loaded, '__file__', 'built-in'))
    except Exception as exc:
        module_errors[module] = repr(exc)

if module_errors:
    raise ImportError(
        'Runtime modules remain unavailable after bootstrap: ' + json.dumps(module_errors) + '. '
        'If Colab restarted while pinning base dependencies or Torch, rerun Notebook 02a from the first cell.'
    )


## Data And Cache Preflight


In [ ]:
from configs.config import PATHS, CONFIG_HASH, CONFIG

required = {
    'Chapman ZIP': Path(PATHS['zip_path']),
    'cache dir': Path(PATHS['cache_dir']),
    'clean cache': Path(PATHS['data_cache']),
    'extract dir': Path(PATHS['extract_dir']),
    'fold PCA cache': Path(PATHS['cache_dir']) / 'revision_pca_models',
    'fold feature cache': Path(PATHS['cache_dir']) / 'revision_feature_cache',
    'model dir': Path(PATHS['model_dir']),
}
for name, path in required.items():
    size = path.stat().st_size if path.is_file() else None
    size_gib = f'{size / (1024 ** 3):.2f} GiB' if size is not None else '-'
    if path.is_dir():
        npz_count = len(list(path.glob('*.npz')))
        joblib_count = len(list(path.glob('*.joblib')))
        print(f'{name:18s}: exists=True npz_count={npz_count} joblib_count={joblib_count} path={path}')
    else:
        print(f'{name:18s}: exists={path.exists()} size={size_gib} path={path}')
# Expected feature-cache names are fingerprinted from the cleaned cache subject order.
# If these files exist, train/evaluation will load them instead of regenerating MiniRocket/HRV.
def inspect_expected_feature_caches():
    import numpy as np
    from src.provenance import record_order_fingerprint

    clean_cache_path = Path(PATHS['data_cache'])
    if not clean_cache_path.is_file():
        print('Feature cache preflight: clean cache is missing, so exact MiniRocket/HRV fingerprint cannot be derived yet.')
        return

    with np.load(clean_cache_path, allow_pickle=True) as payload:
        subjects = payload['subjects']
        stored_fingerprint = (
            str(payload['record_order_fingerprint'].item())
            if 'record_order_fingerprint' in payload.files
            else None
        )
    computed_fingerprint = record_order_fingerprint(subjects)
    if stored_fingerprint and stored_fingerprint != computed_fingerprint:
        raise RuntimeError(
            'Clean cache fingerprint mismatch: '
            f'stored={stored_fingerprint} computed={computed_fingerprint}'
        )
    fingerprint = stored_fingerprint or computed_fingerprint
    n_records = len(subjects)
    cache_dir = Path(PATHS['cache_dir'])
    expected_feature_caches = {
        'expected RAW MiniRocket cache': cache_dir / f'rocket_raw_N{n_records}_C12_L5000_K10000_S42_R{fingerprint}.npz',
        'expected HRV36 cache': cache_dir / f'hrv36_N{n_records}_C12_L5000_R{fingerprint}.npz',
    }
    print('Feature cache fingerprint:', fingerprint)
    all_present = True
    for name, path in expected_feature_caches.items():
        exists = path.is_file()
        all_present = all_present and exists
        size = path.stat().st_size if exists else None
        size_gib = f'{size / (1024 ** 3):.2f} GiB' if size is not None else '-'
        print(f'{name:32s}: exists={exists} size={size_gib} path={path}')
    if all_present:
        print('Feature cache preflight: MiniRocket and HRV36 caches are present and should be reused.')
    else:
        print('Feature cache preflight: at least one feature cache is missing; that feature will be regenerated once and then cached.')

inspect_expected_feature_caches()

if not Path(PATHS['zip_path']).is_file():
    raise FileNotFoundError(f'Missing Chapman ZIP: {PATHS["zip_path"]}')

print('Use clean cache:', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('Save clean cache:', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('Local extract :', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('Config hash:', CONFIG_HASH)
print('Folds      :', CONFIG['n_folds'])
print('Epochs     :', CONFIG['epochs'])
print('Batch size :', CONFIG['batch_size'])
print('Aggregation:', CONFIG['aggregation_method'], 'q=', CONFIG['power_mean_q'])


## Run Five-Fold Retraining


In [ ]:
from datetime import datetime, timezone
import subprocess

RUN_RETRAIN = os.environ.get('ECG_RAMBA_RUN_RETRAIN', '0') == '1'
RESUME_TRAINING = os.environ.get('ECG_RAMBA_RESUME_TRAINING', '1') == '1'
os.environ['ECG_RAMBA_RESUME_TRAINING'] = '1' if RESUME_TRAINING else '0'

log_dir = Path('reports/revision/logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / 'retrain_best_ema_train.log'
durable_log_path = MODEL_DIR / 'retrain_best_ema_train.log'
cmd = [sys.executable, '-u', 'scripts/train.py']
print('Training log (active):', log_path)
print('Training log (durable Drive):', durable_log_path)
existing_ckpts = sorted(MODEL_DIR.glob('fold*_*.pt'))
if RUN_RETRAIN and existing_ckpts and not RESUME_TRAINING:
    preview = ', '.join(path.name for path in existing_ckpts[:8])
    raise RuntimeError(
        f'Model run directory already contains {len(existing_ckpts)} checkpoint files: {MODEL_DIR}. '
        f'Preview: {preview}. Enable RESUME_TRAINING or select a new ECG_RAMBA_RETRAIN_RUN_ID.'
    )
if existing_ckpts and RESUME_TRAINING:
    print(f'Resume enabled: {len(existing_ckpts)} existing checkpoint files will be audited by scripts/train.py.')
print('$', ' '.join(cmd))

if RUN_RETRAIN:
    log_mode = 'a' if RESUME_TRAINING and log_path.exists() else 'w'
    durable_mode = 'a' if RESUME_TRAINING and durable_log_path.exists() else 'w'
    with log_path.open(log_mode, encoding='utf-8') as log, durable_log_path.open(durable_mode, encoding='utf-8') as durable_log:
        if log_mode == 'a':
            log.write('\n--- resumed run ---\n')
        if durable_mode == 'a':
            durable_log.write('\n--- resumed run ---\n')
        header = 'started_utc=' + datetime.now(timezone.utc).isoformat() + '\n' + 'model_dir=' + str(MODEL_DIR) + '\n'
        log.write(header)
        durable_log.write(header)
        log.flush()
        durable_log.flush()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            durable_log.write(line)
            log.flush()
            durable_log.flush()
        returncode = proc.wait()
        footer = 'finished_utc=' + datetime.now(timezone.utc).isoformat() + '\n' + 'returncode=' + str(returncode) + '\n'
        log.write(footer)
        durable_log.write(footer)
        log.flush()
        durable_log.flush()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
    print('Retraining finished:', log_path)
else:
    print('RUN_RETRAIN=False; skipping training and leaving existing artifacts untouched. Set ECG_RAMBA_RUN_RETRAIN=1 or RUN_RETRAIN=True only when retraining is intended.')


## Verify Explicit Checkpoint Contract


In [ ]:
import json
import hashlib
import pandas as pd
import torch

model_dir = Path(os.environ['ECG_RAMBA_MODEL_DIR'])
expected = []
for fold in range(1, 6):
    expected.extend([
        model_dir / f'fold{fold}_best_ema.pt',
        model_dir / f'fold{fold}_best_raw.pt',
        model_dir / f'fold{fold}_final_ema.pt',
        model_dir / f'fold{fold}_final_raw.pt',
    ])
expected.extend([
    model_dir / 'folds.pkl',
    model_dir / 'training_log_epochs.csv',
    model_dir / 'cv_results_clean_core.csv',
    model_dir / 'fold_label_prevalence.csv',
    model_dir / 'fold_split_audit.json',
    model_dir / 'resume_integrity_audit.json',
    model_dir / 'resume_integrity_audit.csv',
])
missing = [str(path) for path in expected if not path.exists()]
if missing:
    raise FileNotFoundError('Missing retrain artifacts: ' + '; '.join(missing))

rows = []
dataset_record_fingerprints = set()
for fold in range(1, 6):
    for checkpoint_kind in ['final_ema', 'best_ema']:
        ckpt = model_dir / f'fold{fold}_{checkpoint_kind}.pt'
        payload = torch.load(ckpt, map_location='cpu')
        if payload.get('weights_kind') != 'ema':
            raise RuntimeError(f'{ckpt} does not report weights_kind=ema')
        expected_rule = 'fixed_final_epoch' if checkpoint_kind == 'final_ema' else 'max_validation_f1_macro'
        if payload.get('selection_rule') != expected_rule:
            raise RuntimeError(f'{ckpt} selection_rule={payload.get("selection_rule")} expected={expected_rule}')
        if checkpoint_kind == 'final_ema' and int(payload.get('epoch', -1)) != RETRAIN_EPOCHS:
            raise RuntimeError(f'{ckpt} epoch does not match RETRAIN_EPOCHS={RETRAIN_EPOCHS}')
        dataset_record_fingerprint = payload.get('dataset_record_order_fingerprint')
        if not dataset_record_fingerprint:
            raise RuntimeError(f'{ckpt} lacks dataset_record_order_fingerprint')
        dataset_record_fingerprints.add(str(dataset_record_fingerprint))
        h = hashlib.sha256(ckpt.read_bytes()).hexdigest()
        rows.append({
            'fold': fold,
            'checkpoint_kind': checkpoint_kind,
            'manuscript_role': 'primary_fixed_epoch' if checkpoint_kind == 'final_ema' else 'diagnostic_validation_selected',
            'path': str(ckpt),
            'sha256': h,
            'epoch': payload.get('epoch'),
            'f1_macro': payload.get('f1_macro'),
            'weights_kind': payload.get('weights_kind'),
            'metrics_weights_kind': payload.get('metrics_weights_kind'),
            'selection_rule': payload.get('selection_rule'),
            'config_hash': payload.get('config_hash'),
            'dataset_record_order_fingerprint': dataset_record_fingerprint,
            'aggregation': payload.get('aggregation'),
            'pca_explained_variance': payload.get('pca_explained_variance'),
            'training_protocol': payload.get('training_protocol'),
        })

if len(dataset_record_fingerprints) != 1:
    raise RuntimeError(f'Checkpoint dataset fingerprints differ: {sorted(dataset_record_fingerprints)}')

manifest_payload = {
    'model_run_id': MODEL_RUN_ID,
    'model_dir': str(model_dir),
    'legacy_model_dir': str(LEGACY_MODEL_DIR),
    'current_model_pointer': str(CURRENT_MODEL_POINTER),
    'durable_training_log': str(MODEL_DIR / 'retrain_best_ema_train.log'),
    'overwrite_legacy_model_dir': False,
    'epochs': RETRAIN_EPOCHS,
    'dataset_record_order_fingerprint': next(iter(dataset_record_fingerprints)),
    'checkpoints': rows,
}
manifest_path = Path('reports/revision/manifests/retrain_best_ema_checkpoint_manifest.json')
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest_payload, indent=2, sort_keys=True), encoding='utf-8')
CURRENT_MODEL_POINTER.parent.mkdir(parents=True, exist_ok=True)
CURRENT_MODEL_POINTER.write_text(str(model_dir), encoding='utf-8')
print('Wrote:', manifest_path)
print('Wrote current model pointer:', CURRENT_MODEL_POINTER, '->', model_dir)
display(pd.DataFrame(rows))

cv_path = model_dir / 'cv_results_clean_core.csv'
train_log_path = model_dir / 'training_log_epochs.csv'
prevalence_path = model_dir / 'fold_label_prevalence.csv'
resume_audit_path = model_dir / 'resume_integrity_audit.json'
resume_audit_csv_path = model_dir / 'resume_integrity_audit.csv'
cv_results = pd.read_csv(cv_path)
train_log = pd.read_csv(train_log_path)
resume_audit = json.loads(resume_audit_path.read_text(encoding='utf-8'))
resume_rows = pd.read_csv(resume_audit_csv_path)
if sorted(resume_audit.get('completed_folds_reused', [])) != sorted(resume_rows.loc[resume_rows['complete_for_resume_skip'].astype(bool), 'fold'].astype(int).tolist()):
    raise RuntimeError('Resume integrity audit JSON/CSV disagree about reused folds.')
if len(train_log) != 5 * RETRAIN_EPOCHS:
    raise RuntimeError(f'Expected {5 * RETRAIN_EPOCHS} epoch rows, found {len(train_log)}')
for fold in range(1, 6):
    fold_log = train_log[train_log['fold'] == fold]
    best_rows = fold_log[fold_log['is_best_epoch'].astype(bool)]
    if len(best_rows) != 1 or best_rows.iloc[0]['validation_weights_kind'] != 'ema':
        raise RuntimeError(f'Fold {fold} must have exactly one EMA best-epoch row; found {len(best_rows)}')

print('CV results:', cv_path)
display(cv_results)
print('Resume integrity audit:', resume_audit_path)
print('Reused folds:', resume_audit.get('completed_folds_reused'))
display(resume_rows)
print('Training log tail:')
display(train_log.tail(10))
print('Fold prevalence audit:', prevalence_path)
display(pd.read_csv(prevalence_path).head(10))

near_boundary = cv_results['best_ema_epoch'] >= (RETRAIN_EPOCHS - 1)
n_near_boundary = int(near_boundary.sum())
print(f'Best epoch within final two epochs: {n_near_boundary}/5 folds')
if RETRAIN_EPOCHS == 20 and n_near_boundary >= 3:
    print('Diagnostic recommendation: a separate e30 sensitivity run may test convergence, but do not replace the pre-specified e20 manuscript OOF by choosing the better result on these same folds.')
else:
    print('No automatic evidence that the epoch horizon is too short; decide from OOF and fold-level curves.')


## Publish Retraining Provenance

Checkpoints and the live training log already reside in `model_runs`; this cell verifies and mirrors the revision manifest/log.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/retrain_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{MIRROR_REVISION_ROOT}"',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/retrain_mirror_publish.log',
)
print('Durable model run:', MODEL_DIR)
print('Canonical revision mirror:', MIRROR_REVISION_ROOT)
print('Retraining provenance publish complete.')
